# Bước 07: Đánh Giá Duy Nhất Một Lần Trên Tập Test Niêm Phong
Dự án: Tốt nghiệp - Energy Forecasting - Nhóm thực hiện: The Outliers
Nguồn tham chiếu: `srcs/05_machine_learning/Forcasting_v3/11_evaluate_final_test.py`

**LƯU Ý VỀ CẤU HÌNH BỘ ĐẶC TRƯNG (`FEATURE_SET_NAME`):**
- Đổi hằng số `FEATURE_SET_NAME = "no_lag1"` ở Cell 2 để chấm phiên bản mới (không lag_1).
- Hoặc đổi `FEATURE_SET_NAME = ""` nếu muốn chấm phiên bản cũ (có đầy đủ lag_1) để so sánh đối chứng.

**NGUYÊN TẮC NIÊM PHONG TẬP TEST:**
- Chấm duy nhất 1 lần trên tập Test niêm phong (`v3_test_selected.parquet`) sau khi đã chọn được hàm loss chiến thắng từ tập Validation.

## Bước 2. Import thư viện và khai báo tham số (Cấu hình FEATURE_SET_NAME)

In [ ]:
import gc
import json
import os
import pickle
import platform
import time
import warnings

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import seaborn as sns
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

# ── HẰNG SỐ CẤU HÌNH BỘ ĐẶC TRƯNG ──
# Đổi thành "" nếu muốn dùng kết quả phiên bản cũ (có lag_1)
FEATURE_SET_NAME = "no_lag1"

VERSION = 'v3'
SITE_COL = 'site_id'
TIMESTAMP_COL = 'timestamp'
TARGET_COL = 'energy_generated_kwh'
HORIZONS = [1, 4]

USE_GPU = True
GPU_PLATFORM_ID = 0
GPU_DEVICE_ID = 0

SELECTED_DIR = '../../data/model/v3/05_selected'
TRAIN_BASE_DIR = '../../data/model/v3/06_train'
OUTPUT_DIR = '../../data/model/v3/07_final_test'
os.makedirs(OUTPUT_DIR, exist_ok=True)
for h in HORIZONS:
    os.makedirs(f'{OUTPUT_DIR}/h{h}', exist_ok=True)

print("Đã import thư viện và khai báo tham số cho Notebook 07.")
print(f"- Feature set name : '{FEATURE_SET_NAME}'")
print(f"- Horizons đánh giá: {HORIZONS}")
print(f"- Ghi kết quả ra   : {OUTPUT_DIR}")

## Bước 2.1. Thiết lập GPU (OpenCL) cho LightGBM

In [ ]:
LA_LINUX = (os.name == "posix" and platform.system() == "Linux")

if LA_LINUX:
    OCL_CANDIDATES = [
        "/run/opengl-driver/etc/OpenCL/vendors",
        "/etc/OpenCL/vendors",
    ]
    if "OCL_ICD_VENDORS" not in os.environ:
        for _p in OCL_CANDIDATES:
            if os.path.isdir(_p) and any(f.endswith(".icd") for f in os.listdir(_p)):
                os.environ["OCL_ICD_VENDORS"] = _p
                print("Đã tự đặt OCL_ICD_VENDORS = " + str(_p))
                break


def kiem_tra_gpu():
    try:
        X = np.random.rand(100, 4)
        y = np.random.rand(100)
        lgb.train(
            {"objective": "regression", "device": "gpu", "gpu_platform_id": GPU_PLATFORM_ID, "gpu_device_id": GPU_DEVICE_ID, "verbose": -1},
            lgb.Dataset(X, y),
            num_boost_round=1
        )
        return True, ""
    except Exception as e:
        return False, str(e)[:200]

GPU_SAN_SANG = False
if USE_GPU:
    GPU_SAN_SANG, _err = kiem_tra_gpu()
    if GPU_SAN_SANG:
        print("GPU OpenCL sẵn sàng. LightGBM sẽ chạy trên GPU.")
    else:
        print("[CẢNH BÁO] Không dùng được GPU, tự động chuyển sang CPU.")
        print("Lý do: " + str(_err))

print("Chế độ tính toán chính thức: " + ("GPU" if GPU_SAN_SANG else "CPU"))

## Bước 3. Đọc kết quả Validation và Chọn Mô hình Chiến thắng cho h1 và h4

In [ ]:
loss_list = ['mse', 'mae', 'huber']
winning_models_summary = {}

for h in HORIZONS:
    h_label = f"h{h}"
    h_desc = "t+1 (15m tới)" if h == 1 else "t+4 (1h tới)"
    val_summary = []

    print("")
    print("=" * 80)
    print(f"--- SO SÁNH LOSS VÀ CHỌN MÔ HÌNH THẮNG FOR {h_label.upper()} ({h_desc}) [FEATURE_SET='{FEATURE_SET_NAME}'] ---")
    print("=" * 80)

    for l_name in loss_list:
        folder_name = f"{l_name}_{FEATURE_SET_NAME}" if FEATURE_SET_NAME else l_name
        val_json_path = f'{TRAIN_BASE_DIR}/{folder_name}/{h_label}/metrics_val.json'

        if not os.path.exists(val_json_path):
            val_json_path = f'{TRAIN_BASE_DIR}/{folder_name}/metrics_val.json'

        if not os.path.exists(val_json_path):
            print(f"[THÔNG BÁO] Chưa có file metrics_val.json tại: {val_json_path}, bỏ qua.")
            continue

        with open(val_json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        if h_label in data:
            data = data[h_label]

        meas_day_m = data.get('measured_daylight', data.get('scope_c_measured_daylight_official', {}))
        meas_m = data.get('measured', data.get('scope_b_measured', {}))
        all_m = data.get('all', data.get('scope_a_all', {}))

        val_summary.append({
            'horizon': h_label,
            'loss_name': l_name.upper(),
            'feature_set': FEATURE_SET_NAME,
            'pooled_wape_cv_%': data.get('pooled_wape_cv'),
            'val_measured_daylight_wape_%': meas_day_m.get('wape'),
            'val_measured_daylight_rmse': meas_day_m.get('rmse'),
            'val_measured_daylight_mae': meas_day_m.get('mae'),
            'val_measured_daylight_r2': meas_day_m.get('r2'),
        })

    if not val_summary:
        raise FileNotFoundError("Chưa có kết quả train nào cho feature_set='" + str(FEATURE_SET_NAME) + "' tại " + TRAIN_BASE_DIR + ". Hãy chạy notebook 06 tương ứng!")

    df_val_comp = pd.DataFrame(val_summary)
    display(df_val_comp)

    best_idx = df_val_comp['val_measured_daylight_wape_%'].idxmin()
    best_loss_name = df_val_comp.loc[best_idx, 'loss_name'].lower()
    best_val_wape = df_val_comp.loc[best_idx, 'val_measured_daylight_wape_%']

    rationale_str = f"Horizon {h_label.upper()}: Loss '{best_loss_name.upper()}' ({FEATURE_SET_NAME}) đạt WAPE tốt nhất trên tập Validation ở phạm vi Measured & Daylight: {best_val_wape:.2f}%."
    print(f"=> LOSS THẮNG DÀNH CHO {h_label.upper()}: {best_loss_name.upper()}")

    winning_models_summary[h_label] = {
        'winning_loss': best_loss_name,
        'feature_set': FEATURE_SET_NAME,
        'rationale': rationale_str,
        'val_wape': best_val_wape,
        'comparison': val_summary,
    }

# Ghi file best_loss.json tổng hợp
with open(f'{OUTPUT_DIR}/best_loss.json', 'w', encoding='utf-8') as f:
    json.dump(winning_models_summary, f, ensure_ascii=False, indent=2)

print("")
print(f"Đã lưu kết quả chọn loss thắng cho cả 2 horizon vào: {OUTPUT_DIR}/best_loss.json")

## Bước 4. Đánh giá Duy nhất Một Lần trên Tập Test Niêm Phong cho cả h1 và h4

In [ ]:
def add_horizon_target(df, horizon_steps):
    out = df.copy()
    target_col_name = f'target_h{horizon_steps}'
    if horizon_steps == 1:
        out[target_col_name] = out[TARGET_COL]
    else:
        shift_steps = -(horizon_steps - 1)
        out[target_col_name] = out.groupby(SITE_COL)[TARGET_COL].shift(shift_steps)
    return out, target_col_name


test_path = f'{SELECTED_DIR}/{VERSION}_test_selected.parquet'
print(f"Đang đọc tập Test niêm phong từ: {test_path}")
test_raw = pd.read_parquet(test_path)

if 'has_complete_history_features' in test_raw.columns:
    test_base = test_raw[test_raw['has_complete_history_features'] == True].copy()
else:
    test_base = test_raw.copy()
del test_raw
gc.collect()

test_audit_df = test_base[[SITE_COL, TIMESTAMP_COL, 'energy_source', 'is_daylight']].copy() if 'is_daylight' in test_base.columns else test_base[[SITE_COL, TIMESTAMP_COL]].copy()

for h in HORIZONS:
    h_label = f"h{h}"
    h_desc = "t+1 (15m tới)" if h == 1 else "t+4 (1h tới)"
    h_dir = f"{OUTPUT_DIR}/{h_label}"
    os.makedirs(h_dir, exist_ok=True)

    win_loss = winning_models_summary[h_label]['winning_loss']
    folder_name = f"{win_loss}_{FEATURE_SET_NAME}" if FEATURE_SET_NAME else win_loss

    win_model_dir = f'{TRAIN_BASE_DIR}/{folder_name}/{h_label}'
    if not os.path.exists(f'{win_model_dir}/model.pkl'):
        win_model_dir = f'{TRAIN_BASE_DIR}/{folder_name}'

    with open(f'{win_model_dir}/model.pkl', 'rb') as f:
        model = pickle.load(f)
    with open(f'{win_model_dir}/model_config.json', 'r', encoding='utf-8') as f:
        m_config = json.load(f)

    features = m_config['features']
    feature_medians = pd.Series(m_config['feature_medians'], dtype=float)

    test_h, t_col = add_horizon_target(test_base, h)
    scored = test_h[test_h[t_col].notna()].copy()

    feat_cols = [c for c in features if c in scored.columns]
    num_cols = [c for c in feat_cols if pd.api.types.is_numeric_dtype(scored[c])]
    feat_cols = num_cols

    X_test = scored[feat_cols].fillna(feature_medians).astype(float)
    y_true = scored[t_col].values
    y_pred = model.predict(X_test)

    scored['y_true'] = y_true
    scored['y_pred'] = y_pred
    scored['residual'] = y_true - y_pred

    if 'lag_1' in scored.columns:
        scored['y_pred_baseline'] = scored['lag_1'].values
    else:
        scored['y_pred_baseline'] = np.nan

    def compute_wape_func(yt, yp):
        abs_y = np.nansum(np.abs(yt))
        return (np.nansum(np.abs(yt - yp)) / abs_y * 100.0) if abs_y > 0 else np.nan

    def compute_metrics_func(yt, yp):
        return {
            'wape': compute_wape_func(yt, yp),
            'rmse': root_mean_squared_error(yt, yp),
            'mae': mean_absolute_error(yt, yp),
            'r2': r2_score(yt, yp),
        }

    m_test_a = compute_metrics_func(y_true, y_pred)
    wape_base_a = compute_wape_func(y_true, scored['y_pred_baseline'].values) if 'lag_1' in scored.columns else np.nan
    imprv_a = ((wape_base_a - m_test_a['wape']) / wape_base_a * 100.0) if pd.notna(wape_base_a) and wape_base_a > 0 else np.nan

    mask_meas = (scored['energy_source'] == 'measured').values if 'energy_source' in scored.columns else np.ones(len(scored), dtype=bool)
    m_test_b = compute_metrics_func(y_true[mask_meas], y_pred[mask_meas]) if mask_meas.sum() > 0 else {'wape': np.nan, 'rmse': np.nan, 'mae': np.nan, 'r2': np.nan}
    wape_base_b = compute_wape_func(y_true[mask_meas], scored['y_pred_baseline'].values[mask_meas]) if ('lag_1' in scored.columns and mask_meas.sum() > 0) else np.nan
    imprv_b = ((wape_base_b - m_test_b['wape']) / wape_base_b * 100.0) if pd.notna(wape_base_b) and wape_base_b > 0 else np.nan

    if 'is_daylight' in scored.columns:
        mask_day = (scored['is_daylight'] == True).values | (scored['is_daylight'] == 1).values
        mask_meas_day = mask_meas & mask_day
    else:
        mask_meas_day = np.zeros(len(scored), dtype=bool)

    m_test_c = compute_metrics_func(y_true[mask_meas_day], y_pred[mask_meas_day]) if mask_meas_day.sum() > 0 else {'wape': np.nan, 'rmse': np.nan, 'mae': np.nan, 'r2': np.nan}
    wape_base_c = compute_wape_func(y_true[mask_meas_day], scored['y_pred_baseline'].values[mask_meas_day]) if ('lag_1' in scored.columns and mask_meas_day.sum() > 0) else np.nan
    imprv_c = ((wape_base_c - m_test_c['wape']) / wape_base_c * 100.0) if pd.notna(wape_base_c) and wape_base_c > 0 else np.nan

    print("")
    print(f"================ BÁO CÁO ĐÁNH GIÁ TẬP TEST FOR {h_label.upper()} ({h_desc}) [FEATURE_SET='{FEATURE_SET_NAME}'] ================")
    print(f"Mô hình thắng: {win_loss.upper()}")
    print(f"(a) ALL     : WAPE {m_test_a['wape']:.2f}% | Base {wape_base_a:.2f}% | Cải thiện {imprv_a:.2f}%")
    print(f"(b) MEASURED: WAPE {m_test_b['wape']:.2f}% | Base {wape_base_b:.2f}% | Cải thiện {imprv_b:.2f}%")
    print(f"(c) MEASURED & DAYLIGHT (CHÍNH THỨC): WAPE {m_test_c['wape']:.2f}% | Base {wape_base_c:.2f}% | Cải thiện {imprv_c:.2f}%")

    metrics_h_payload = {
        'horizon_steps': h,
        'winning_loss': win_loss,
        'feature_set_name': FEATURE_SET_NAME,
        'total_test_rows': len(scored),
        'measured_test_rows': int(mask_meas.sum()),
        'measured_daylight_test_rows': int(mask_meas_day.sum()),
        'all': {**m_test_a, 'baseline_persistence_wape': wape_base_a, 'improvement_vs_baseline_pct': imprv_a},
        'measured': {**m_test_b, 'baseline_persistence_wape': wape_base_b, 'improvement_vs_baseline_pct': imprv_b},
        'measured_daylight': {**m_test_c, 'baseline_persistence_wape': wape_base_c, 'improvement_vs_baseline_pct': imprv_c},
    }
    with open(f'{h_dir}/metrics_overall.json', 'w', encoding='utf-8') as f:
        json.dump(metrics_h_payload, f, ensure_ascii=False, indent=2)

    # Metrics by site
    site_rows = []
    use_mask = mask_meas_day if mask_meas_day.sum() > 0 else (mask_meas if mask_meas.sum() > 0 else np.ones(len(scored), dtype=bool))
    scored_site = scored[use_mask].copy()
    for s_id, grp in scored_site.groupby(SITE_COL, observed=True):
        sm = compute_metrics_func(grp['y_true'].values, grp['y_pred'].values)
        site_rows.append({'site_id': s_id, 'rows': len(grp), **sm})
    df_site = pd.DataFrame(site_rows)
    df_site.to_csv(f'{h_dir}/metrics_by_site.csv', index=False)

    # Ghép vào audit dataframe chung
    audit_sub = scored[[SITE_COL, TIMESTAMP_COL, 'y_true', 'y_pred', 'residual']].copy()
    audit_sub = audit_sub.rename(columns={
        'y_true': f'y_true_h{h}',
        'y_pred': f'y_pred_h{h}',
        'residual': f'residual_h{h}',
    })
    test_audit_df = test_audit_df.merge(audit_sub, on=[SITE_COL, TIMESTAMP_COL], how='left')

# Ghi file prediction_audit.parquet
test_audit_df.to_parquet(f'{OUTPUT_DIR}/prediction_audit.parquet', index=False)
print("")
print(f"Đã xuất file dự báo tổng hợp cả h1 và h4: {OUTPUT_DIR}/prediction_audit.parquet")

## Bước 5. Trực quan hóa Kết quả và So sánh h1 vs h4

In [ ]:
print("--- TRỰC QUAN HÓA KẾT QUẢ DỰ BÁO TẬP TEST (CẢ h1 t+1 VÀ h4 t+4) ---")
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

sample_site = test_audit_df[SITE_COL].iloc[0]
sample_data = test_audit_df[(test_audit_df[SITE_COL] == sample_site)].sort_values(TIMESTAMP_COL).head(200)

if 'y_true_h1' in sample_data.columns and 'y_pred_h1' in sample_data.columns:
    axes[0, 0].plot(sample_data[TIMESTAMP_COL], sample_data['y_true_h1'], label='Thực tế (y_true)', color='blue', alpha=0.7)
    axes[0, 0].plot(sample_data[TIMESTAMP_COL], sample_data['y_pred_h1'], label='Dự báo h1 (t+1: 15m)', color='orange', linestyle='--')
if 'y_pred_h4' in sample_data.columns:
    axes[0, 0].plot(sample_data[TIMESTAMP_COL], sample_data['y_pred_h4'], label='Dự báo h4 (t+4: 1h)', color='green', linestyle=':')
axes[0, 0].set_title(f'Dự báo h1 & h4 vs Thực tế (no_lag1) tại Trạm: {sample_site}')
axes[0, 0].set_xlabel('Thời gian')
axes[0, 0].set_ylabel('Sản lượng (kWh)')
axes[0, 0].legend()

if 'residual_h1' in test_audit_df.columns:
    sns.histplot(test_audit_df['residual_h1'].dropna(), kde=True, ax=axes[0, 1], color='orange', label='Residual h1 (t+1)', bins=50, alpha=0.5)
if 'residual_h4' in test_audit_df.columns:
    sns.histplot(test_audit_df['residual_h4'].dropna(), kde=True, ax=axes[0, 1], color='green', label='Residual h4 (t+4)', bins=50, alpha=0.5)
axes[0, 1].set_title('So sánh Phân bố Sai số Residuals (h1 vs h4)')
axes[0, 1].set_xlabel('Sai số (kWh)')
axes[0, 1].legend()

if os.path.exists(f'{OUTPUT_DIR}/h1/metrics_by_site.csv'):
    df_s1 = pd.read_csv(f'{OUTPUT_DIR}/h1/metrics_by_site.csv')
    sns.barplot(data=df_s1, x='site_id', y='rmse', ax=axes[1, 0], palette='Blues_d')
    axes[1, 0].set_title('RMSE theo Trạm cho h1 (t+1: 15m)')
    axes[1, 0].set_ylabel('RMSE (kWh)')
    axes[1, 0].tick_params(axis='x', rotation=45)

if os.path.exists(f'{OUTPUT_DIR}/h4/metrics_by_site.csv'):
    df_s4 = pd.read_csv(f'{OUTPUT_DIR}/h4/metrics_by_site.csv')
    sns.barplot(data=df_s4, x='site_id', y='rmse', ax=axes[1, 1], palette='Greens_d')
    axes[1, 1].set_title('RMSE theo Trạm cho h4 (t+4: 1h)')
    axes[1, 1].set_ylabel('RMSE (kWh)')
    axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### TỔNG KẾT BỘ ĐẶC TRƯNG NO_LAG1 VÀ NGUYÊN TẮC ĐỐI CHỨNG
1. **Loại bỏ trễ pha:** Bằng cách loại bỏ `lag_1`, mô hình buộc phải dự báo dựa trên các đặc trưng vật lý bức xạ thời tiết và các chu kỳ dài hơn (`lag_4`, `lag_96`, `rolling_*`).
2. **So sánh đối chứng:** Bạn có thể thay đổi hằng số `FEATURE_SET_NAME = ""` ở Cell 2 để nạp lại kết quả phiên bản cũ (có `lag_1`) và so sánh đối chiếu trực tiếp chỉ số WAPE giữa 2 phương án.